In [1]:
pip install rapidfuzz pandas


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import unicodedata
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz

In [3]:
mis_form_path = "../../backend/source/forms"
flow_form_path = "./output/flow_forms"

flow_ids = {
    "8520967": 1749634736797, # WTP
    "17260923": 1748903240763, # WWTP
    "27040920": 1749611049520, # SPS (Pump Station)
    "1520924": 1749623934933, # EPS Water quality
    "5530933": 1749623934933, # EPS Project Construction
    "2490944": 1749621221728, # RWS
}

meta_config = {
    "flow": [
        "name",
        "surveyId",
        "surveyGroupId",
        "version",
        "app",
        ["questionGroup", "heading"],
        ["questionGroup", "repeatable"],
    ],
    "mis": [
        "id",
        "form",
        "type",
        "version",
        ["question_groups", "label"],
        ["question_groups", "repeatable"],
    ],
}

# Configuration - Text-only matching (ignoring question groups completely)
SIMILARITY_CONFIG = {
    "text_similarity_threshold": 80,  # Minimum score for text match (0-100)
    "use_group_weighting": False,     # Completely ignore question groups
    "group_weight": 0.0,              # No group weighting
    "text_weight": 1.0,               # Use only text similarity
}

In [4]:
def flatten_flow_form_pd(
    json_path: Path,
    record_path: str = "questionGroup.question",
    meta_type: str = "flow",
) -> pd.DataFrame:
    if meta_type not in meta_config:
        raise ValueError(f"meta_type must be one of {list(meta_config.keys())}")
    # read top-level JSON as a pandas Series (one object)
    j = pd.read_json(json_path, typ="series").to_dict()

    # normalize nested questions: record_path points to question lists inside questionGroup
    df = pd.json_normalize(
        j,
        record_path=record_path.split("."),
        meta=meta_config[meta_type],
        meta_prefix=f"{meta_type}_",
        errors="ignore"
    )
    # rename question id column for clarity
    if "id" in df.columns:
        df = df.rename(
            columns={
                "id": f"{meta_type}_question_id"
            }
        )

    # rename group meta columns for clarity
    if "flow_questionGroup.heading" in df.columns:
        df = df.rename(
            columns={
                "flow_questionGroup.heading": "flow_question_group",
                "flow_questionGroup.repeatable": "flow_question_group_repeatable"
            }
        )

    if "mis_question_groups.label" in df.columns:
        df = df.rename(
            columns={
                "mis_question_groups.label": "mis_question_group",
                "mis_question_groups.repeatable": "mis_question_group_repeatable"
            }
        )

    return df

In [5]:
def _normalize_alpha_only(val):
    if pd.isna(val):
        return ""
    s = str(val)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    # remove anything that's not a-z (keeps only letters)
    s = re.sub(r"[^a-z]+", "", s)
    return s

In [6]:
def _normalize_for_similarity(val):
    """
    Normalize text for similarity matching.
    Keeps more structure than _normalize_alpha_only for better fuzzy matching.
    """
    if pd.isna(val):
        return ""
    s = str(val)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    # Remove extra whitespace and normalize punctuation
    s = re.sub(r'\s+', ' ', s)
    s = s.strip()
    return s

In [7]:
def calculate_text_similarity(text1: str, text2: str) -> float:
    """
    Calculate text similarity score using rapidfuzz.
    Uses partial_ratio which is good for matching substrings.
    
    This function completely ignores question groups.
    
    Args:
        text1: First text string (Flow question label)
        text2: Second text string (MIS question label)
    
    Returns:
        Similarity score between 0 and 100
    """
    norm_text1 = _normalize_for_similarity(text1)
    norm_text2 = _normalize_for_similarity(text2)
    
    # Use partial_ratio for better matching of questions with slight variations
    # Example: "Which Division-Province-Tikina are you in?" vs "Which Division-Province-Tikina?"
    text_score = fuzz.partial_ratio(norm_text1, norm_text2)
    
    return text_score

In [8]:
def find_best_match_by_text(
    flow_text: str,
    mis_questions_df: pd.DataFrame,
    config: dict = SIMILARITY_CONFIG
) -> tuple[pd.DataFrame, dict]:
    """
    Find the best matching MIS question for a Flow question based ONLY on text similarity.
    Question groups are completely ignored.
    
    Args:
        flow_text: Flow question label text
        mis_questions_df: DataFrame containing MIS questions
        config: Configuration dictionary
    
    Returns:
        tuple: (matched_row, match_info)
    """
    if mis_questions_df.empty:
        return pd.DataFrame(), {"matched": False, "reason": "No MIS questions available"}
    
    # Get all MIS question labels
    mis_labels = mis_questions_df["label"].tolist()
    
    best_score = 0
    best_idx = None
    
    # Iterate through all MIS questions to find best text match
    for idx, mis_label in enumerate(mis_labels):
        score = calculate_text_similarity(flow_text, mis_label)
        
        if score > best_score:
            best_score = score
            best_idx = idx
    
    # Check if best score meets threshold
    if best_score >= config["text_similarity_threshold"]:
        match_info = {
            "matched": True,
            "score": best_score,
            "method": "text_similarity"
        }
        return mis_questions_df.iloc[[best_idx]], match_info
    else:
        match_info = {
            "matched": False,
            "score": best_score,
            "reason": f"Score {best_score:.1f} below threshold {config['text_similarity_threshold']}"
        }
        return pd.DataFrame(), match_info

In [9]:
def match_questions_by_text(
    flow_questions_df: pd.DataFrame,
    mis_questions_df: pd.DataFrame,
    config: dict = SIMILARITY_CONFIG
) -> pd.DataFrame:
    """
    Match Flow questions to MIS questions using text-only similarity matching.
    Question groups are completely ignored.
    
    This function replaces the exact matching logic with a fuzzy matching approach
    that works on question labels only.
    """
    rows = []
    unmatched_count = 0
    low_confidence_count = 0
    
    # Iterate every unique flow question row grouped by surveyId, group, question id
    for (survey_id, group, qid), grp in flow_questions_df.groupby(
        ["flow_surveyId", "flow_question_group", "flow_question_id"], dropna=False
    ):
        rep = grp.iloc[0]
        f_text = rep.get("text", "")
        f_group = rep.get("flow_question_group", "")
        
        # Find best match using text similarity only (ignoring groups)
        matched_df, match_info = find_best_match_by_text(f_text, mis_questions_df, config)
        
        if match_info["matched"]:
            # Collect aggregated values from matched rows
            def agg_col(df, col):
                if col not in df.columns or df.empty:
                    return ""
                vals = df[col].dropna().astype(str).unique()
                return ";".join(vals)
            
            mis_id_val = agg_col(matched_df, "mis_id")
            mis_qgroup_val = agg_col(matched_df, "mis_question_group")
            mis_qid_val = agg_col(matched_df, "mis_question_id")
            label_val = agg_col(matched_df, "label")
            
            # Determine confidence level based on score
            score = match_info["score"]
            if score >= 95:
                confidence = "high"
            elif score >= 85:
                confidence = "medium"
            else:
                confidence = "low"
                low_confidence_count += 1
            
            rows.append({
                "flow_form_id": survey_id,
                "flow_question_group": group,
                "flow_question_label": f_text,
                "flow_question_id": qid,
                "mis_form_id": mis_id_val,
                "mis_question_group": mis_qgroup_val,
                "mis_question_label": label_val,
                "mis_question_id": mis_qid_val,
                "match_score": round(score, 2),
                "match_confidence": confidence,
                "match_method": match_info["method"]
            })
        else:
            # No match found - still create a row with empty MIS values
            rows.append({
                "flow_form_id": survey_id,
                "flow_question_group": group,
                "flow_question_label": f_text,
                "flow_question_id": qid,
                "mis_form_id": "",
                "mis_question_group": "",
                "mis_question_label": "",
                "mis_question_id": "",
                "match_score": round(match_info.get("score", 0), 2),
                "match_confidence": "none",
                "match_method": "none"
            })
            unmatched_count += 1
    
    mapping_df = pd.DataFrame(rows)
    
    # Print summary statistics
    total_questions = len(mapping_df)
    matched_questions = total_questions - unmatched_count
    print(f"\n=== Matching Summary ===")
    print(f"Total Flow questions: {total_questions}")
    print(f"Successfully matched: {matched_questions} ({matched_questions/total_questions*100:.1f}%)")
    print(f"Unmatched: {unmatched_count} ({unmatched_count/total_questions*100:.1f}%)")
    print(f"Low confidence matches: {low_confidence_count}")
    print(f"Threshold: {config['text_similarity_threshold']}%")
    
    return mapping_df

In [10]:
# Main processing loop
for json_form in filter(
    lambda f: f.suffix == ".json",
    Path(flow_form_path).iterdir(),
):
    print(f"\n{'='*60}")
    print(f"Loading form from {json_form}")
    
    # Get filename and extract form id
    flow_form_id = json_form.name.split("_")[0]
    print(f"Form ID: {flow_form_id}")
    
    # Load Flow questions
    flow_questions_df = flatten_flow_form_pd(json_form)
    
    # Get MIS form id from mapping
    mis_form_id = flow_ids.get(flow_form_id)
    
    # Find MIS form JSON
    mis_form_json = next(filter(
        lambda f: f.suffix == ".json" and str(mis_form_id) in f.name,
        Path(mis_form_path).iterdir(),
    ), None)
    
    if mis_form_json is None:
        print(f"No matching MIS form found for Flow form ID {flow_form_id} and MIS form ID {mis_form_id}")
        continue
    
    print(f"Matching MIS form found: {mis_form_json.name}")
    
    # Load MIS questions
    mis_questions_df = flatten_flow_form_pd(
        json_path=mis_form_json,
        record_path="question_groups.questions",
        meta_type="mis",
    )
    
    # Load child MIS forms
    for child_form_json in filter(
        lambda f: f.suffix == ".json",
        Path(mis_form_path).iterdir(),
    ):
        j = pd.read_json(child_form_json, typ="series").to_dict()
        if j.get("parent_id") == mis_form_id:
            print(f"Loading child MIS form: {child_form_json.name}")
            child_mis_questions_df = flatten_flow_form_pd(
                json_path=child_form_json,
                record_path="question_groups.questions",
                meta_type="mis",
            )
            mis_questions_df = pd.concat([mis_questions_df, child_mis_questions_df], ignore_index=True)
    
    # Normalize question text and group headings
    flow_questions_df["_norm"] = flow_questions_df.get("text", "").apply(_normalize_alpha_only)
    mis_questions_df["_norm"] = mis_questions_df.get("label", "").apply(_normalize_alpha_only)
    
    flow_questions_df["_group_norm"] = flow_questions_df.get("flow_question_group", "").apply(_normalize_alpha_only)
    mis_questions_df["_group_norm"] = mis_questions_df.get("mis_question_group", "").apply(_normalize_alpha_only)
    
    # Use text-only similarity matching (completely ignoring question groups)
    mapping_df = match_questions_by_text(flow_questions_df, mis_questions_df)
    
    # Guarantee uniqueness
    mapping_df = mapping_df.drop_duplicates(
        subset=["flow_form_id", "flow_question_group", "flow_question_id"],
        keep="first"
    ).reset_index(drop=True)
    
    # Save mapping CSV
    output_csv = f"../../backend/source/akvo-flow/{flow_form_id}_forms_mapping.csv"
    mapping_df.to_csv(output_csv, index=False)
    print(f"\nSaved {len(mapping_df)} mapping rows to {output_csv}")


Loading form from output/flow_forms/17260923_WAF_Wastewater_Treatment_Plant_Inspection.json
Form ID: 17260923
Matching MIS form found: 5_1748903240763.prod.json
Loading child MIS form: 5_1748905550055.monitoring.prod.json
Loading child MIS form: 5_1748918946591.monitoring.prod.json

=== Matching Summary ===
Total Flow questions: 103
Successfully matched: 89 (86.4%)
Unmatched: 14 (13.6%)
Low confidence matches: 0
Threshold: 80%

Saved 103 mapping rows to ../../backend/source/akvo-flow/17260923_forms_mapping.csv

Loading form from output/flow_forms/2490944_Rural_Water_Project_Inspection.json
Form ID: 2490944
Matching MIS form found: 3_1749621221728.prod.json
Loading child MIS form: 3_1749621962296.monitoring.prod.json
Loading child MIS form: 3_1749631041125.monitoring.prod.json

=== Matching Summary ===
Total Flow questions: 39
Successfully matched: 28 (71.8%)
Unmatched: 11 (28.2%)
Low confidence matches: 2
Threshold: 80%

Saved 39 mapping rows to ../../backend/source/akvo-flow/2490944_